In [ ]:
%reload_ext autoreload
%autoreload 2
from importlib import reload

import os
import sys

import logging
import warnings
import numpy as np
import astropy as ap
import scipy as sp
import scipy.stats
import matplotlib as mpl
import matplotlib.pyplot as plt

import h5py
import tqdm.notebook as tqdm
import pickle

import kalepy as kale
import kalepy.utils
import kalepy.plot

import holodeck as holo
import holodeck.sams
import holodeck.gravwaves
from holodeck import cosmo, utils, plot, discrete, sams, host_relations, _PATH_DATA
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC
from pathlib import Path

import compare_sams

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
#mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
#mpl.rc('lines', solid_capstyle='round')
#mpl.rc('mathtext', fontset='cm')
#plt.rcParams.update({'grid.alpha': 0.5})
#mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode

log = holo.log
log.setLevel(logging.INFO)


NLOUD = 1
##NREALS = 500
NREALS = 10


# PRODUCTION RUNS
- galaxy pars use dev defaults (no mmbulge evol, no ng15)
- new hardening model type 0
- $\alpha_{char} = -0.5$
- $r_{char,9} = 10^{0.5}pc$
- param sweeps for: $a_{gw,9}$, $\nu_{inner}$, $\alpha_{gw}$, $\beta_{gw}$, $r_{char,9}$, & $\tau_{out}$.
- NLOUD = 5
- NREALS = 100
- gridshape = 100
- also trying an alternate fiducial model with beta=+1/4 (const inner tscale for all mass ratios)

# Alternate fiducial model 
### ($\beta_{gw}=+1/4$, $\alpha_{gw}=-1/4$, $\alpha_{char}=-1/2$, $\nu_{inner}=0$, $a_{gw,9}=10^{2.5} R_g$)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = +0.25
_rch9_dflt = 10.0**0.5 # pc
_alphch_dflt = -0.5 # not varying

NLOUD = 5
NREALS = 100
_gridshape = 100 
TAU = None

# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
fpath = '/Users/lblecha/nanograv/gensams/gensams_nreal100/fiducial_beta0pt25_devdflt_mod0_nui0'
GPFFLAG = None

# sparse or full param sweep?
sparse_sweep = False


# ---- r9 var ----
#tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
#                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
#                               sparse_param_sweep=sparse_sweep,
#                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
#                               gpfflag=GPFFLAG, tau=TAU,
#                               _tout_default=_tout_dflt, 
#                               _nuin_default=_nui_dflt, 
#                               _rgw9_default=None, 
#                               _alphgw_default=_alphgw_dflt, 
#                               _betagw_default=_betagw_dflt,
#                               _rch9_default=_rch9_dflt,                               
#                               _alphch_default=_alphch_dflt,
#                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)


##### Fiducial model 
### ($\alpha_{gw}=-1/4$, $\alpha_{char}=-1/2$, $\nu_{inner}=0$, $a_{gw,9}=10^{2.5} R_g$)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 10.0**0.5 # pc
_alphch_dflt = -0.5 # not varying

NLOUD = 5
NREALS = 100
_gridshape = 100 
TAU = None

# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
fpath = '/Users/lblecha/nanograv/gensams/gensams_nreal100/fiducial_devdflt_mod0_nui0'
GPFFLAG = None

# sparse or full param sweep?
sparse_sweep = False


# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)


# 'Simple' Model 
### ($\alpha_{gw}=0$, $\alpha_{char}=0$, $\nu_{inner}=0$, $a_{gw,9}=10^{2.5} R_g$)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = 0.0
_betagw_dflt = 0.0
_rch9_dflt = 10.0**0.5 # pc
_alphch_dflt = 0.0 # not varying

NLOUD = 5
NREALS = 100
_gridshape = 100 
TAU = None

# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
fpath = '/Users/lblecha/nanograv/gensams/gensams_nreal100/simple_devdflt_mod0_nui0'
GPFFLAG = None

# sparse or full param sweep?
sparse_sweep = False

# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)


# 'Star-like' Model w/ fiducial $\alpha$ values
### ($\alpha_{gw}=-1/4$, $\alpha_{char}=-1/2$, $\nu_{inner}=-1$, $a_{gw,9}=10^{3.5} R_g$)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = -1.0
_rgw9_dflt = 10.0**3.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 10.0**0.5 # pc
_alphch_dflt = -0.5 # not varying

NLOUD = 5
NREALS = 100
_gridshape = 100 
TAU = None

# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
fpath = '/Users/lblecha/nanograv/gensams/gensams_nreal100/star_fiducial_devdflt_mod0_nui0'
GPFFLAG = None

# sparse or full param sweep?
sparse_sweep = False


# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)


# 'Gas-like' Model w/ fiducial $\alpha$ values
### ($\alpha_{gw}=-1/4$, $\alpha_{char}=-1/2$, $\nu_{inner}=+2$, $a_{gw,9}=10^{2} R_g$)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = +2.0
_rgw9_dflt = 10.0**2 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 10.0**0.5 # pc
_alphch_dflt = -0.5 # not varying

NLOUD = 5
NREALS = 100
_gridshape = 100 
TAU = None

# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
fpath = '/Users/lblecha/nanograv/gensams/gensams_nreal100/gas_fiducial_devdflt_mod0_nui0'
GPFFLAG = None

# sparse or full param sweep?
sparse_sweep = False


# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               sparse_param_sweep=sparse_sweep,
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)


# older stuff

# Model 0, $\alpha_{\rm char} = -0.5$, dev defaults with KH13 z_plaw=1.0

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 
TAU = None
# ---- dev-defaults-KH13evol:
_gal_pars = 'dev-defaults-KH13evol' 
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/kh13evol_mod0_nui0_alphach-0pt5'
GPFFLAG = None
## ---- dev-defaults:
#_gal_pars = 'dev-defaults' 
#fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/devdflt_mod0_nui0_alphach-0pt5'
#GPFFLAG = None
# ---- NG15:
#_gal_pars = 'NG15' 
#GPFFLAG = 1
#fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui0_alphach-0pt5'


# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)


# Model 0, $\alpha_{\rm char} = -0.5$, dev defaults

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 
TAU = None
# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/devdflt_mod0_nui0_alphach-0pt5'
GPFFLAG = None
# ---- NG15:
#_gal_pars = 'NG15' 
#GPFFLAG = 1
#fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui0_alphach-0pt5'

# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)



## 'Star-like' Model 0 dev-defaults, defaults: $r_{\rm gw,9}=10^{3.5}R_g$, $\nu_{\rm in}=-1$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = -1.0
_rgw9_dflt = 10.0**3.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 
TAU = None
# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
GPFFLAG = None
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/devdflt_mod0_nui-1_alphach-0pt5'
# ---- NG15:
#_gal_pars = 'NG15' 
#GPFFLAG = 1
#fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui-1_alphach-0pt5'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## 'Gas-like' Model 0, dev-defaults, defaults: $r_{\rm gw,9}=10^{2}R_g$, $\nu_{\rm in}=2$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = +2.0
_rgw9_dflt = 10.0**2 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 
TAU = None
# ---- dev-defaults:
_gal_pars = 'dev-defaults' 
GPFFLAG = None
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/devdflt_mod0_nui2_alphach-0pt5'
# ---- NG15:
#_gal_pars = 'NG15' 
#GPFFLAG = 1
#fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui2_alphach-0pt5'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# Model 0,  $\alpha_{\rm char} = -0.5$, NG15 gal pars

## Model 0, defaults: $r_{\rm gw,9}=10^{2.5}R_g$, $\nu_{\rm in}=0$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 # new to 2nd version
TAU = None
# ---- dev-defaults:
#_gal_pars = 'dev-defaults' 
#GPFFLAG = None
# ---- NG15:
_gal_pars = 'NG15' # new to 2nd version
GPFFLAG = 1
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui0_alphach-0pt5'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## 'Star-like' Model 0, defaults: $r_{\rm gw,9}=10^{3.5}R_g$, $\nu_{\rm in}=-1$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = -1.0
_rgw9_dflt = 10.0**3.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 # new to 2nd version
TAU = None
# ---- dev-defaults:
#_gal_pars = 'dev-defaults' 
#GPFFLAG = None
# ---- NG15:
_gal_pars = 'NG15' # new to 2nd version
GPFFLAG = 1
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui-1_alphach-0pt5'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## 'Gas-like' Model 0, defaults: $r_{\rm gw,9}=10^{2}R_g$, $\nu_{\rm in}=2$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = +2.0
_rgw9_dflt = 10.0**2 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 # new to 2nd version
TAU = None
# ---- dev-defaults:
#_gal_pars = 'dev-defaults' 
#GPFFLAG = None
# ---- NG15:
_gal_pars = 'NG15' # new to 2nd version
GPFFLAG = 1
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui2_alphach-0pt5'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# No-mass-dependence version of Model 0

## No-mass-dependence Model 0, defaults: $r_{\rm gw,9}=10^{2.5}R_g$, $\nu_{\rm in}=0$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = 0$, $\alpha_{GW}=0$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = 0.0
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = 0.0 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 # new to 2nd version
TAU = None
# ---- dev-defaults:
#_gal_pars = 'dev-defaults' 
#GPFFLAG = None
# ---- NG15:
_gal_pars = 'NG15' # new to 2nd version
GPFFLAG = 1
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui0_nomassdep'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## Star-like No-mass-dependence Model 0, defaults: $r_{\rm gw,9}=10^{3.5}R_g$, $\nu_{\rm in}=-1$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = 0$, $\alpha_{GW}=0$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = -1.0
_rgw9_dflt = 10.0**3.5 # Rg
_alphgw_dflt = 0.0
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = 0.0 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 # new to 2nd version
TAU = None
# ---- dev-defaults:
#_gal_pars = 'dev-defaults' 
#GPFFLAG = None
# ---- NG15:
_gal_pars = 'NG15' # new to 2nd version
GPFFLAG = 1
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui-1_nomassdep'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## Gas-like No-mass-dependence Model 0, defaults: $r_{\rm gw,9}=10^{2}R_g$, $\nu_{\rm in}=2$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = 0$, $\alpha_{GW}=0$

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = +2.0
_rgw9_dflt = 10.0**2 # Rg
_alphgw_dflt = 0.0
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = 0.0 # not varying this param for now

NLOUD = 5
NREALS = 10
_gridshape = 100 # new to 2nd version
TAU = None
# ---- dev-defaults:
#_gal_pars = 'dev-defaults' 
#GPFFLAG = None
# ---- NG15:
_gal_pars = 'NG15' # new to 2nd version
GPFFLAG = 1
fpath = '/Users/lblecha/nanograv/gensams/gensams_060426/ng15_mod0_nui2_nomassdep'

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=fpath, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# OLD VERSIONS - no subdir specified at runtime

## Model 0, defaults: $r_{\rm gw,9}=10^{2.5}R_g$, $\nu_{\rm in}=0$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -2/3$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
1. first version stored in `/Users/lblecha/nanograv/gensams/gensams_v052126/mod0_nui0_rgw91e2pt5_newalph_rchar`
2. 2nd version with NG15 pars for non-hardening stuff stored in `/Users/lblecha/nanograv/gensams/gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar`
3. 3rd version: testing new setup with `dev-defaults` setting, should match first version 

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -2/3.0 # not varying this param for now

NLOUD = 5
NREALS = 10
#_gal_pars = 'NG15' # new to 2nd version
_gal_pars = 'dev-defaults' # 3rd version
_gridshape = 100 # new to 2nd version
#GPFFLAG = 1 # new to 2nd version
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## Model 0, defaults: $r_{\rm gw,9}=10^{2}R_g$, $\nu_{\rm in}=+2$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -2/3$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
1. first version stored in `/Users/lblecha/nanograv/gensams/gensams_v052126/mod0_nui2_rgw91e2_newalph_rchar`
2. 2nd version with NG15 pars for non-hardening stuff stored in `/Users/lblecha/nanograv/gensams/gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar`

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = +2.0
_rgw9_dflt = 10.0**2 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -2/3.0 # not varying this param for now
_gal_pars = 'NG15' # new to 2nd version
_gridshape = 100 # new to 2nd version

NLOUD = 5
NREALS = 10
GPFFLAG = 1 # new to 2nd version
#GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## Model 0, defaults: $r_{\rm gw,9}=10^{3.5}R_g$, $\nu_{\rm in}=-1$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -2/3$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
1. first version stored in `/Users/lblecha/nanograv/gensams/gensams_v052126/mod0_nui-1_rgw91e3pt5_newalph_rchar`
2. 2nd version with NG15 pars for non-hardening stuff stored in `/Users/lblecha/nanograv/gensams/gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar`

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = -1.0
_rgw9_dflt = 10.0**3.5 # Rg
_alphgw_dflt = -0.25
_betagw_dflt = 0.0
_rch9_dflt = 3.0 # pc
_alphch_dflt = -2/3.0 # not varying this param for now
_gal_pars = 'NG15' # new to 2nd version
_gridshape = 100 # new to 2nd version

NLOUD = 5
NREALS = 10
GPFFLAG = 1 # new to 2nd version
#GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _betagw_default=_betagw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               _gal_pars_type=_gal_pars, gridshape=_gridshape,
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=_betagw_dflt,
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## NG15-like model with old hardening

In [ ]:
# ---- SET DEFAULT PARAMS ----
_tout_dflt = None
_nui_dflt = -0.5
_rgw9_dflt = None
_alphgw_dflt = None
_betagw_dflt = None
_rch9_dflt = None
_alphch_dflt = None
_gal_pars = 'NG15' 
_gridshape = 100 

NLOUD = 5
NREALS = 10
GPFFLAG = 1 # required for NG15-like

# ---- 2PL hardening with ph15 hardening vals and NG15 approximate posterior values for galaxy pars ----
for TAU in [0.1, 0.5, 1.0, 2.0]:
    tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                                   suite_type='ph15_ng15', hard_type='fixed2PL',
                                   _gal_pars_type=_gal_pars, gridshape=_gridshape,
                                   gpfflag=GPFFLAG, tau=TAU,
                                   _tout_default=_tout_dflt,
                                   _nuin_default=_nui_dflt, 
                                   _rgw9_default=_rgw9_dflt, 
                                   _alphgw_default=_alphgw_dflt,
                                   _betagw_default=_betagw_dflt,
                                   _rch9_default=_rch9_dflt,
                                   _alphch_default=_alphch_dflt,
                                   pickle_sams=True, pickle_name=None)

## Model 0: testing the rchar9 and alpha_char parameterization 

### rgw9 default = 10^2.5 Rg, nu_inner default = 0

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_rch9_dflt = 100.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

In [ ]:
# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

### rgw9 default = 10^3 Rg, nu_inner default = 0

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**3 # Rg
_alphgw_dflt = -0.25
_rch9_dflt = 100.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

### rgw9 default = 10^2 Rg, nu_inner default = 0

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2 # Rg
_alphgw_dflt = -0.25
_rch9_dflt = 100.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

### rgw9 default = 10^2.5 Rg, nu_inner default = 0.5

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = +0.5
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_rch9_dflt = 100.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

### rgw9 default = 10^2.5 Rg, nu_inner default = -0.5

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = -0.5
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_rch9_dflt = 100.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=None,
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=None,                                
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt, 
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rgw9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=None, 
                               _alphgw_default=_alphgw_dflt, 
                               _rch9_default=_rch9_dflt,                               
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphgwvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _rgw9_default=_rgw9_dflt, 
                               _alphgw_default=None, 
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rch9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=None,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

### Model 0: rgw9 default = 10^2.5 Rg, nu_inner default = 0.0, testing beta_gw_crit variation

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**2.5 # Rg
_alphgw_dflt = -0.25
_rch9_dflt = 100.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

### Model 0: rgw9 default = 10^3 Rg, nu_inner default = 0.0, testing beta_gw_crit variation

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_rgw9_dflt = 10.0**3.0 # Rg
_alphgw_dflt = -0.25
_rch9_dflt = 100.0 # pc
_alphch_dflt = -0.5 # not varying this param for now

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- beta var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_betagwvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, 
                               _nuin_default=_nui_dflt, 
                               _alphgw_default=_alphgw_dflt, 
                               _betagw_default=None, 
                               _rgw9_default=_rgw9_dflt,
                               _rch9_default=_rch9_dflt,
                               _alphch_default=_alphch_dflt,
                               pickle_sams=True, pickle_name=None)

## Standard default for model 1 param sweep (r9 = 10^2.5 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_dadt_dflt = -1.0e6 # cm/s
_alph_dflt = -0.25
_r9_dflt = 10.0**2.5 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- dadt var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_dadtvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

## alternate default for model 1 param sweep (r9 = 100 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_dadt_dflt = -1.0e6 # cm/s
_alph_dflt = -0.25
_r9_dflt = 10.0**2.0 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- dadt var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_dadtvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

## alternate defaults for model 1 param sweep (r9 = 1000 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_dadt_dflt = -1.0e6 # cm/s
_alph_dflt = -0.25
_r9_dflt = 10.0**3.0 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- dadt var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_dadtvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

## alternate defaults for model 1 param sweep (r9 = 10^3.5 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_dadt_dflt = -1.0e6 # cm/s
_alph_dflt = -0.25
_r9_dflt = 10.0**3.5 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- dadt var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_dadtvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

# Model 0

# standard default for model 0 param sweep (r9 = 10^2.5 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_alph_dflt = -0.25
_r9_dflt = 10.0**2.5 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
#tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
#                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
#                               gpfflag=GPFFLAG, tau=TAU,
#                               _nuin_default=_nui_dflt, _alph_default=_alph_dflt,
#                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
#                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

# alternate default for model 0 param sweep (r9 = 10^3 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_alph_dflt = -0.25
_r9_dflt = 10.0**3.0 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=_nui_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

# alternate default for model 0 param sweep (r9 = 10^2 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_alph_dflt = -0.25
_r9_dflt = 10.0**2.0 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=_nui_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

# standard default for model 0 param sweep (r9 = 10^3.5 Rg)

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_alph_dflt = -0.25
_r9_dflt = 10.0**3.5 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

# ---- tout var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=_nui_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- nuin var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _alph_default=_alph_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt, 
                               pickle_sams=True, pickle_name=None)
# ---- r9 var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _rch_default=_rch_dflt,                               
                               pickle_sams=True, pickle_name=None)

# ---- alpha var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# ---- rchar var ----
tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_rchvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _tout_default=_tout_dflt, _nuin_default=_nui_dflt, 
                               _alph_default=_alph_dflt, _r9_default=_r9_dflt,
                               pickle_sams=True, pickle_name=None)

### set up single param sweep; create SAMs and pickle

In [ ]:
# ---- SET DEFAULT PARAMS FOR PARAM SWEEP ----
_tout_dflt = 1.0 # Gyr
_nui_dflt = 0.0
_dadt_dflt = -1.0e6 # cm/s
_alph_dflt = -0.25
_r9_dflt = 10.0**2.5 # Rg
_rch_dflt = 100.0 # pc

NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type1_toutvar', hard_type='fixedOuter',
                               gpfflag=GPFFLAG, tau=TAU,
                               _dadt_default=_dadt_dflt, _alph_default=_alph_dflt,
                               _r9_default=_r9_dflt, _rch_default=_rch_dflt,
                               pickle_sams=True, pickle_name=None)

# old stuff

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_toutvar', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.0, _alph_default=-0.25,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 100
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', gpfflag=GPFFLAG, tau=TAU,
                               _alph_default=-0.25,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.0, _alph_default=-0.25,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-1alph-025')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.0,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.5, _alph_default=-0.25,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-15alph-025')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=0.0, _alph_default=-0.25,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui0alph-025')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.0, _alph_default=-0.5,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-1alph-05')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.0, _alph_default=0.0,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-1alph-0')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.5, _alph_default=-0.5,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-15alph-05')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.5, _alph_default=0.0,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-15alph0')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=0.0, _alph_default=-0.5,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui0alph-05')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=0.0, _alph_default=0.0,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui0alph0')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-0.5, _alph_default=-0.5,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-05alph-05')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-0.5, _alph_default=-0.25,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-05alph-025')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_r9var', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-0.5, _alph_default=0,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='_nui-05alph0')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', gpfflag=GPFFLAG, tau=TAU,
                               _alph_default=-0.25, _r9_default=300,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='alph-025r9300')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', gpfflag=GPFFLAG, tau=TAU,
                               _alph_default=-0.25, _r9_default=100,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='alph-025r9100')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', gpfflag=GPFFLAG, tau=TAU,
                               _alph_default=-0.25, _r9_default=30,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='alph-025r930')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', gpfflag=GPFFLAG, tau=TAU,
                               _alph_default=-0.5, _r9_default=100,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='alph-05r9100')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_nuivar', gpfflag=GPFFLAG, tau=TAU,
                               _alph_default=0.0, _r9_default=100,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='alph0r9100')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.0, _r9_default=100,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='nui-1r9100')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=0.0, _r9_default=1000,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='nui0r91e3')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=-1.5, _r9_default=1000,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='nui-15r91e3')

In [ ]:
NLOUD = 5
NREALS = 10
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='new_hardening_type0_alphvar', gpfflag=GPFFLAG, tau=TAU,
                               _nuin_default=0.0, _r9_default=100,
                               pickle_sams=True, pickle_name=None, pickle_name_extra='nui0r9100')

In [ ]:
NLOUD = 5
NREALS = 500
GPFFLAG = 0
TAU = 1.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='manual', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)


In [ ]:
NLOUD = 5
NREALS = 500
GPFFLAG = 0
TAU = 3.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='manual', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)


In [ ]:
NLOUD = 5
NREALS = 500
GPFFLAG = 1
TAU = 5.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='manual', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)


In [ ]:
NLOUD = 5
NREALS = 50
GPFFLAG = None
TAU = None

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='old_new_mods_compare', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)


In [ ]:
NLOUD = 5
NREALS = 50
GPFFLAG = None
TAU = 1.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='old_new_mods_compare', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 50
GPFFLAG = None
TAU = 11.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='old_new_mods_compare', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 50
GPFFLAG = None
TAU = 0.1

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='old_new_mods_compare', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 50
GPFFLAG = None
TAU = 3.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='old_new_mods_compare', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 50
GPFFLAG = None
TAU = 10.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='old_new_mods_compare', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)

In [ ]:
NLOUD = 5
NREALS = 50
GPFFLAG = None
TAU = 8.0

tmp = compare_sams.create_sams(nreals=NREALS, nloud=NLOUD, fpath=_PATH_DATA, 
                               suite_type='old_new_mods_compare', gpfflag=GPFFLAG, tau=TAU,
                               pickle_sams=True, pickle_name=None)

In [ ]:
#print("creating 'no-GPF' SAM using Galaxy Merger Rate (GMR) "
#      "    (uses galaxy merger rates directly from RG15 instead of GPF+GMT)...")
#sam_gmr = sams.Semi_Analytic_Model()

#print("    ...calculating hardening for no-GPF (GMR) SAM")
##default values I've been running with: rchar=10.0*PC, gamma_inner=-1.0, gamma_outer=+1.5
#hard_gmr = holo.hardening.Fixed_Time_2PL_SAM(sam_gmr, tau_hard*GYR, sepa_init=1.0e4*PC)

#print("    ...creating gwb for no-GPF (GMR) SAM")
#### ***NOTE*** this returns hc, not hc2, & has no pars in function:
#gwb_new_sam_gmr = sam_gmr.gwb_new(freqs_edges, hard_gmr, realize=NREALS) 
#### ***NOTE*** this one allows for including pars of loud sources:
#gwb_sam_gmr = sam_gmr.gwb(freqs_edges, hard_gmr, realize=NREALS, loudest=NLOUD, params=True)


##default values I've been running with: rchar=10.0*PC, gamma_inner=-1.0, gamma_outer=+1.5
#gmr_pkl_fname = (f"sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{tau_hard}"
#                 f"_rchar{rchar}_nuin{gamma_inner}_nuout{gamma_outer}_gmr.pkl")
##gmr_pkl_fname = f"sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{tau_hard}_gmr.pkl"
#gmr_data = sam_gmr, hard_gmr, gwb_new_sam_gmr, gwb_sam_gmr, freqs, freqs_edges
#with open(gmr_pkl_fname, "wb") as f:
#    pickle.dump(gmr_data, f)


## ---- Define the GWB frequencies
#freqs, freqs_edges = utils.pta_freqs()
#print(f"{freqs.shape[0]=}, {freqs_edges.shape[0]=}")
#NFREQS = freqs.shape[0]
#NREALS = 500
#NLOUD = 10
##NLOUD = 1

## ---- Create discrete population(s) (including a rescaled version of TNG300, rTN300
##tmp = compare_discrete.create_dpops(allow_mbh0=True, mod_mmbulge=True, skip_evo=False, fsa_only=False, 
##                                    inclRescale=True, nreals=NREALS, nloudest=NLOUD, fpath=_SIM_MERGER_PATH)
##all_dpops, tng_dpops, all_fsa_dpops, tng_fsa_dpops = tmp
##tmp = compare_discrete.create_dpops(allow_mbh0=True, mod_mmbulge=True, skip_evo=False, fsa_only=True, 
##                                    inclRescale=True, nreals=NREALS, nloudest=NLOUD, bfrac=[0.5,1.0], 
##                                    subhalo_mstar_defn='SubhaloMassType', fpath=_SIM_MERGER_PATH)
##all_dpops, tng_dpops, all_fsa_dpops, tng_fsa_dpops = tmp


#print("creating SAM using Galaxy Pair Fraction (GPF) + Galaxy Merger Timescale (GMT)...")

#sam = sams.Semi_Analytic_Model(gpf = sams.GPF_Power_Law())

#print("    ...calculating hardening")
#hard = holo.hardening.Fixed_Time_2PL_SAM(sam, all_fsa_dpops[0].tau, sepa_init=1.0e4*PC)

#print("    ...creating gwb")
#gwb_sam = sam.gwb(freqs_edges, hard, realize=NREALS, loudest=NLOUD, params=True)

##gwb_sam_L100 = sam.gwb(freqs_edges, hard, realize=50, loudest=100, params=True)

## this is what we've been using as the default for the sam GWB,
## until we had to switch to get the param output
##gwb_sam = sam.gwb_new(freqs_edges, hard, realize=50)
#gwb_new_sam = sam.gwb_new(freqs_edges, hard, realize=NREALS)  ### ***NOTE*** this returns hc, not hc2

In [ ]:
#print("    ...creating gwb")
#gwb_sam = sam.gwb(freqs_edges, hard, realize=NREALS, loudest=NLOUD, params=True)

#gwb_sam_L100 = sam.gwb(freqs_edges, hard, realize=50, loudest=100, params=True)

# this is what we've been using as the default for the sam GWB,
# until we had to switch to get the param output
#gwb_sam = sam.gwb_new(freqs_edges, hard, realize=50)